<a href="https://colab.research.google.com/github/amanpoonia/SBeta_preparation/blob/main/Portfolio_weighing_schemas_and_factor_exposure.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**MSCI paper**

*How does the choice of weighing schemas affect the factor exposure of the portfolio. *


In [4]:
#AIM:
'''
1. Not to exactly replicate the results, as no access to CRSP/WRDS, so understand
    and replicate the methodology in depth.
'''

'\n1. Not to exactly replicate the results, as no access to CRSP/WRDS, so understand\n    and replicate the methodology in depth.\n'

In [1]:
import numpy as np
import pandas as pd
import yfinance as yf
import scipy.stats as sp
import matplotlib.pyplot as plt

In [2]:
#Single factor to consider
#Value

#Methodology:
'''
1. A universe of stocks of chosen. MSCI world index or in our case due to lack of data resources
    we choose S&P index constituents.

2. Create a measure called FACTOR SCORE.
    This measure will be used to create a ranking.
    Top 30% of this ranking is chosen
    Now 6 different portfolio weighing schemas are applied

'''

'\n1. A universe of stocks of chosen. MSCI world index or in our case due to lack of data resources\n    we choose S&P index constituents.\n2. \n\n'

In [10]:
#ticker list of SPX for year 2026, from refinitiv LSEG
tickers = pd.read_csv('value_factor_sample.csv')


,ticker,date,price,b_p,fwd_e_p,cfo_ev
0,ALPH,2021-05-31,271.22,0.323,0.0417,0.0659
1,ATLS,2021-05-31,173.69,0.634,0.0980,0.1172
2,BOLT,2021-05-31,75.15,0.223,0.0374,0.0372
3,BRIX,2021-05-31,257.20,0.372,0.0767,0.0457
4,CEDR,2021-05-31,267.54,0.408,0.0453,0.0836
...,...,...,...,...,...,...
295,VOLT,2025-11-30,422.83,1.007,0.0868,0.1573
296,WREN,2025-11-30,144.73,1.855,0.1057,0.1229
297,XYLO,2025-11-30,148.74,0.307,0.0277,0.0653
298,YARN,2025-11-30,173.83,1.077,0.0420,0.0890


In [7]:
start_date = '2024-01-01'
end_date = '2026-09-13'

ticker_list = tickers['tickers']
print(ticker_list)
data = yf.download(ticker_list, start=start_date, end=end_date)

KeyError: 'tickers'

In [4]:
data['Close']

Ticker,ALGN,DDOG,FOXA,GE,KVUE,MSI,NTRS,SHW,TMO
Date,,,,,,,,,
2024-01-02,268.920013,115.080002,29.220104,99.026184,19.014225,302.703888,78.684486,297.583801,539.272644
2024-01-03,256.220001,112.419998,28.988201,97.540199,18.749773,303.053528,76.561401,289.639343,524.080078
2024-01-04,261.359985,113.019997,29.046173,98.011963,18.423613,302.810699,76.357430,288.663452,532.543579
2024-01-05,264.679993,115.989998,29.123476,99.293518,18.820295,300.294922,77.386513,289.551514,526.468445
2024-01-08,273.940002,119.250000,29.519648,100.858109,19.067118,303.888885,78.044769,292.274475,536.527527
...,...,...,...,...,...,...,...,...,...
2026-09-04,158.639999,212.929993,65.419998,337.119995,18.519033,466.842773,186.630005,333.709991,613.309937
2026-09-08,153.490005,210.229996,65.230003,334.910004,18.173161,461.785889,184.880005,326.089996,602.558167
2026-09-09,152.919998,225.270004,63.860001,325.420013,17.540709,458.703857,184.690002,320.640015,605.226074


In [5]:
panel = data.stack(level="Ticker", future_stack=True).reset_index()
panel.columns.name = None
panel

,Date,Ticker,Close,High,Low,Open,Volume
0,2024-01-02,ALGN,268.920013,272.859985,266.070007,271.000000,761800
1,2024-01-02,DDOG,115.080002,119.224998,114.278000,119.224998,4431200
2,2024-01-02,FOXA,29.220104,29.480998,28.427760,28.669329,2889600
3,2024-01-02,GE,99.026184,100.755904,98.467963,99.859593,5189425
4,2024-01-02,KVUE,19.014225,19.265456,18.881998,18.908444,35216200
...,...,...,...,...,...,...,...
6079,2026-09-11,KVUE,17.570354,17.753173,17.392477,17.688938,47159300
6080,2026-09-11,MSI,464.957672,469.775201,464.728254,466.673237,690800
6081,2026-09-11,NTRS,189.190002,189.589996,187.070007,188.949997,700800
6082,2026-09-11,SHW,323.309998,324.910004,317.549988,321.559998,2172200


In [7]:
panel_filtered = panel.loc[:, ['Date', 'Ticker', 'Close']]
display(panel_filtered.head(20))

,Date,Ticker,Close
0,2024-01-02,ALGN,268.920013
1,2024-01-02,DDOG,115.080002
2,2024-01-02,FOXA,29.220104
3,2024-01-02,GE,99.026184
4,2024-01-02,KVUE,19.014225
5,2024-01-02,MSI,302.703888
6,2024-01-02,NTRS,78.684486
7,2024-01-02,SHW,297.583801
8,2024-01-02,TMO,539.272644
9,2024-01-03,ALGN,256.220001


In [8]:
t = yf.Ticker('ALGN')
info = t.info
cashflow = t.cashflow

booktoprice = None
fEtoP = None
cfotoev = None

#Get PRICE/BOOK
#for B/P we take inverse of P/B
price_to_book = info.get('priceToBook')
print(price_to_book)
if price_to_book != 0 and price_to_book is not None:
  booktoprice = 1/price_to_book
print(f"B/P : {booktoprice}")

#Get Price/Earning
#for forward E/P we inverse it
price_to_earning = info.get('forwardPE')
print(price_to_earning)
if price_to_earning != 0 and price_to_earning is not None:
  earningtoprice = 1/price_to_earning
print(f"forwardE/P : {earningtoprice}")


#For CFO/EV
#cashflow operating to enterprise value

#Get Operating cashflow
op_cf = None
if not cashflow.empty and 'Operating Cash Flow' in cashflow.index:
  op_cf = cashflow.loc['Operating Cash Flow'][0]

#Get Equity value
ev = info.get('enterpriseValue')
if ev is not None and ev != 0:
  cfo_by_ev = op_cf/ev

print(booktoprice, earningtoprice, cfo_by_ev)


2.472436
B/P : 0.40445940764492994
11.704939
forwardE/P : 0.0854340206300947
0.40445940764492994 0.0854340206300947 0.06275775383245008


/tmp/ipykernel_1685/1784521194.py:32: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  op_cf = cashflow.loc['Operating Cash Flow'][0]


EXPLORATION BELOW


In [25]:
display(cashflow)
j = cashflow.loc['Operating Cash Flow'][0]
print(j)
type(j)

print(info.get('operatingCashflow'))

,2025-12-31,2024-12-31,2023-12-31,2022-12-31,2021-12-31
Free Cash Flow,4.907780e+08,6.226510e+08,608060000.0,2.768320e+08,NaN
Repurchase Of Capital Stock,-4.659390e+08,-3.528780e+08,-592360000.0,-4.350360e+08,NaN
Repayment Of Debt,NaN,NaN,NaN,NaN,0.0
Issuance Of Capital Stock,2.174900e+07,2.528100e+07,26595000.0,2.614900e+07,NaN
Capital Expenditure,-1.024450e+08,-1.155800e+08,-177716000.0,-2.919000e+08,NaN
Income Tax Paid Supplemental Data,1.347530e+08,1.770820e+08,294569000.0,2.318840e+08,NaN
End Cash Position,1.096186e+09,1.044963e+09,938519000.0,9.423550e+08,NaN
Beginning Cash Position,1.044963e+09,9.385190e+08,942355000.0,1.100139e+09,NaN
Effect Of Exchange Rate Changes,3.502500e+07,-2.115300e+07,4671000.0,-1.151400e+07,NaN
Changes In Cash,1.619800e+07,1.275970e+08,-8507000.0,-1.462700e+08,NaN


593223000.0
755696000


/tmp/ipykernel_1908/1950746513.py:2: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  j = cashflow.loc['Operating Cash Flow'][0]


In [ ]:
#GEMINI
financial_data = []

for ticker_symbol in ticker_list:
    ticker = yf.Ticker(ticker_symbol)
    info = ticker.info
    cashflow = ticker.cashflow

    # Initialize with None
    book_to_price = None
    forward_earnings_to_price = None
    cfo_to_ev = None

    # Book-to-Price (B/P)
    # yfinance provides PriceToBook, so B/P is its inverse
    price_to_book = info.get('priceToBook') # 'priceToBook' is the key in yfinance info
    if price_to_book is not None and price_to_book != 0:
        book_to_price = 1 / price_to_book

    # Forward Earnings-to-Price (E/P)
    # yfinance provides forwardPE (Forward P/E), so Fwd E/P is its inverse
    forward_pe = info.get('forwardPE') # 'forwardPE' is the key in yfinance info
    if forward_pe is not None and forward_pe != 0:
        forward_earnings_to_price = 1 / forward_pe

    # Operating Cash Flow-to-Enterprise Value (CFO/EV)
    operating_cash_flow = None
    if not cashflow.empty and 'Operating Cash Flow' in cashflow.index:
        # Get the most recent operating cash flow (assuming current data is the first column)
        operating_cash_flow = cashflow.loc['Operating Cash Flow'].iloc[0]

    enterprise_value = info.get('enterpriseValue') # 'enterpriseValue' is the key in yfinance info
    if operating_cash_flow is not None and enterprise_value is not None and enterprise_value != 0:
        cfo_to_ev = operating_cash_flow / enterprise_value

    financial_data.append({
        'Ticker': ticker_symbol,
        'Book-to-Price': book_to_price,
        'Forward Earnings-to-Price': forward_earnings_to_price,
        'Operating Cash Flow-to-Enterprise Value': cfo_to_ev
    })

financial_df = pd.DataFrame(financial_data)
display(financial_df)

In [30]:
type(cashflow)

pandas.core.frame.DataFrame

In [27]:
print('--- Keys available in the `info` dictionary: ---')
print(info.keys())

print('\n--- Head of the `cashflow` DataFrame: ---')
display(cashflow.head())

print('\n--- Index of the `cashflow` DataFrame (financial statement items): ---')
print(cashflow.index.tolist())

--- Keys available in the `info` dictionary: ---
dict_keys(['address1', 'address2', 'city', 'state', 'zip', 'country', 'phone', 'website', 'industry', 'industryKey', 'industryDisp', 'sector', 'sectorKey', 'sectorDisp', 'longBusinessSummary', 'fullTimeEmployees', 'companyOfficers', 'auditRisk', 'boardRisk', 'compensationRisk', 'shareHolderRightsRisk', 'overallRisk', 'governanceEpochDate', 'compensationAsOfEpochDate', 'irWebsite', 'executiveTeam', 'maxAge', 'priceHint', 'previousClose', 'open', 'dayLow', 'dayHigh', 'regularMarketPreviousClose', 'regularMarketOpen', 'regularMarketDayLow', 'regularMarketDayHigh', 'payoutRatio', 'beta', 'trailingPE', 'forwardPE', 'volume', 'regularMarketVolume', 'averageVolume', 'averageVolume10days', 'averageDailyVolume10Day', 'bid', 'ask', 'bidSize', 'askSize', 'marketCap', 'nonDilutedMarketCap', 'fiftyTwoWeekLow', 'fiftyTwoWeekHigh', 'allTimeHigh', 'allTimeLow', 'fulldayPrice', 'fulldayChange', 'fulldayChangePercent', 'priceToSalesTrailing12Months', 'fif

,2025-12-31,2024-12-31,2023-12-31,2022-12-31,2021-12-31
Free Cash Flow,490778000.0,622651000.0,608060000.0,276832000.0,NaN
Repurchase Of Capital Stock,-465939000.0,-352878000.0,-592360000.0,-435036000.0,NaN
Repayment Of Debt,NaN,NaN,NaN,NaN,0.0
Issuance Of Capital Stock,21749000.0,25281000.0,26595000.0,26149000.0,NaN
Capital Expenditure,-102445000.0,-115580000.0,-177716000.0,-291900000.0,NaN



--- Index of the `cashflow` DataFrame (financial statement items): ---
['Free Cash Flow', 'Repurchase Of Capital Stock', 'Repayment Of Debt', 'Issuance Of Capital Stock', 'Capital Expenditure', 'Income Tax Paid Supplemental Data', 'End Cash Position', 'Beginning Cash Position', 'Effect Of Exchange Rate Changes', 'Changes In Cash', 'Financing Cash Flow', 'Cash Flow From Continuing Financing Activities', 'Net Other Financing Charges', 'Net Common Stock Issuance', 'Common Stock Payments', 'Common Stock Issuance', 'Net Issuance Payments Of Debt', 'Net Long Term Debt Issuance', 'Long Term Debt Payments', 'Investing Cash Flow', 'Cash Flow From Continuing Investing Activities', 'Net Other Investing Changes', 'Net Investment Purchase And Sale', 'Sale Of Investment', 'Purchase Of Investment', 'Net Business Purchase And Sale', 'Purchase Of Business', 'Net PPE Purchase And Sale', 'Purchase Of PPE', 'Operating Cash Flow', 'Cash Flow From Continuing Operating Activities', 'Change In Working Capita

### Calculating Financial Ratios for All Tickers

The following code iterates through each ticker in your `ticker_list` and retrieves its fundamental information using `yfinance`. It then calculates the requested financial ratios: Book-to-Price (B/P), Forward Earnings-to-Price (Fwd E/P), and Operating Cash Flow-to-Enterprise Value (CFO/EV). The results are compiled into a pandas DataFrame for easy viewing.

In [1]:
import numpy as np

historical_financial_data = []

# The panel_filtered data is from 2024-01-01 to 2026-09-13.
# We will collect financial data for fiscal year ends in 2023, 2024, and 2025.
# yfinance provides annual statements where columns are fiscal year-end dates.

for ticker_symbol in ticker_list:
    ticker_yf = yf.Ticker(ticker_symbol)

    # Corrected attribute names for annual data
    annual_financials = ticker_yf.financials
    annual_cashflow = ticker_yf.cashflow
    annual_balance_sheet = ticker_yf.balance_sheet

    # Get available fiscal year-end dates from the annual statements
    # Ensure we only process dates that appear in all three statements for consistency
    # Also, filter for relevant years (2023, 2024, 2025) and ensure date is not after price data end.
    available_dates = annual_financials.columns.intersection(annual_cashflow.columns).intersection(annual_balance_sheet.columns)

    relevant_dates = [
        d for d in available_dates
        if d.year in [2023, 2024, 2025] and d <= pd.Timestamp('2026-09-13') # Ensure date is within or before price data end
    ]
    relevant_dates.sort() # Process chronologically

    for fiscal_year_end_date in relevant_dates:
        # Convert fiscal_year_end_date to a pandas Timestamp for consistent comparison
        fy_end_ts = pd.Timestamp(fiscal_year_end_date)

        # --- Retrieve closing price for the ticker on or around the fiscal year-end date ---
        # We need the price on the fiscal year-end or the last trading day before it.
        temp_price_df = panel_filtered[panel_filtered['Ticker'] == ticker_symbol].set_index('Date').sort_index()

        price_on_fy_end = np.nan
        if not temp_price_df.empty:
            # Reindex the price series to include the target date, then forward fill
            price_series_for_ticker = temp_price_df['Close']
            reindexed_price = price_series_for_ticker.reindex(price_series_for_ticker.index.union([fy_end_ts])).ffill()
            if fy_end_ts in reindexed_price.index:
                price_on_fy_end = reindexed_price.loc[fy_end_ts]
            # Fallback if ffill didn't work for exact date (e.g., date before first trading day, or holiday)
            if pd.isna(price_on_fy_end) and fy_end_ts not in price_series_for_ticker.index:
                reindexed_price = price_series_for_ticker.reindex(price_series_for_ticker.index.union([fy_end_ts])).bfill()
                if fy_end_ts in reindexed_price.index:
                    price_on_fy_end = reindexed_price.loc[fy_end_ts]

        # Initialize ratios as NaN
        book_to_price = np.nan
        earnings_to_price = np.nan
        cfo_to_ev = np.nan

        # --- Calculate Book-to-Price (B/P) ---
        total_stockholder_equity = annual_balance_sheet.loc['Total Stockholder Equity', fy_end_ts] if 'Total Stockholder Equity' in annual_balance_sheet.index else np.nan

        # Shares outstanding for Book Value calculation (at fiscal year end)
        common_stock_shares_outstanding = np.nan
        if 'Common Stock Shares Outstanding' in annual_balance_sheet.index:
            common_stock_shares_outstanding = annual_balance_sheet.loc['Common Stock Shares Outstanding', fy_end_ts]
        elif 'Basic Average Shares' in annual_financials.index:
            common_stock_shares_outstanding = annual_financials.loc['Basic Average Shares', fy_end_ts]
        # Note: Using `info.get('sharesOutstanding')` as a fallback here would be a rough approximation for historical data

        if pd.notna(total_stockholder_equity) and pd.notna(common_stock_shares_outstanding) and \
           common_stock_shares_outstanding > 0 and pd.notna(price_on_fy_end) and price_on_fy_end > 0:
            book_value_per_share = total_stockholder_equity / common_stock_shares_outstanding
            if book_value_per_share > 0:
                book_to_price = book_value_per_share / price_on_fy_end

        # --- Calculate Earnings-to-Price (E/P) ---
        # Using Basic EPS from annual financials for historical E/P
        basic_eps = annual_financials.loc['Basic EPS', fy_end_ts] if 'Basic EPS' in annual_financials.index else np.nan
        if pd.notna(basic_eps) and pd.notna(price_on_fy_end) and price_on_fy_end > 0:
            earnings_to_price = basic_eps / price_on_fy_end

        # --- Calculate Operating Cash Flow-to-Enterprise Value (CFO/EV) ---
        operating_cash_flow = annual_cashflow.loc['Operating Cash Flow', fy_end_ts] if 'Operating Cash Flow' in annual_cashflow.index else np.nan

        # Historical Enterprise Value (EV) calculation: Market Cap + Total Debt - Cash & Equivalents
        total_debt = annual_balance_sheet.loc['Total Debt', fy_end_ts] if 'Total Debt' in annual_balance_sheet.index else np.nan
        cash_and_equivalents = annual_balance_sheet.loc['Cash And Cash Equivalents', fy_end_ts] if 'Cash And Cash Equivalents' in annual_balance_sheet.index else np.nan
        market_cap_fy_end = price_on_fy_end * common_stock_shares_outstanding if pd.notna(price_on_fy_end) and pd.notna(common_stock_shares_outstanding) else np.nan

        historical_enterprise_value = np.nan
        if pd.notna(market_cap_fy_end) and pd.notna(total_debt) and pd.notna(cash_and_equivalents):
            historical_enterprise_value = market_cap_fy_end + total_debt - cash_and_equivalents

        if pd.notna(operating_cash_flow) and pd.notna(historical_enterprise_value) and historical_enterprise_value != 0:
            cfo_to_ev = operating_cash_flow / historical_enterprise_value

        historical_financial_data.append({
            'Date': fy_end_ts,
            'Ticker': ticker_symbol,
            'Book-to-Price': book_to_price,
            'Earnings-to-Price': earnings_to_price,
            'Operating Cash Flow-to-Enterprise Value': cfo_to_ev
        })

financial_df_historical = pd.DataFrame(historical_financial_data)

# Ensure 'Date' column is datetime for merging
financial_df_historical['Date'] = pd.to_datetime(financial_df_historical['Date'])

display(financial_df_historical.head())

NameError: name 'ticker_list' is not defined

### Merging Daily Price Data with Financial Ratios

To combine the daily price data (`panel_filtered`) with the static financial ratios (`financial_df`), we will perform a `merge` operation using the common 'Ticker' column. This will attach the calculated financial ratios to each corresponding daily stock entry.

In [2]:
merged_df = pd.merge(panel_filtered, financial_df, on='Ticker', how='left')
display(merged_df.head())

NameError: name 'pd' is not defined

In [9]:
print(tickers['tickers'])

KeyError: 'tickers'

WORK ON THIS DATASET

In [3]:
#ticker list of SPX for year 2026, from refinitiv LSEG
data = pd.read_csv('value_factor_sample.csv')

In [4]:
data

,ticker,date,price,b_p,fwd_e_p,cfo_ev
0,ALPH,2021-05-31,271.22,0.323,0.0417,0.0659
1,ATLS,2021-05-31,173.69,0.634,0.0980,0.1172
2,BOLT,2021-05-31,75.15,0.223,0.0374,0.0372
3,BRIX,2021-05-31,257.20,0.372,0.0767,0.0457
4,CEDR,2021-05-31,267.54,0.408,0.0453,0.0836
...,...,...,...,...,...,...
295,VOLT,2025-11-30,422.83,1.007,0.0868,0.1573
296,WREN,2025-11-30,144.73,1.855,0.1057,0.1229
297,XYLO,2025-11-30,148.74,0.307,0.0277,0.0653
298,YARN,2025-11-30,173.83,1.077,0.0420,0.0890


In [6]:
# Convert 'date' column to datetime objects
data['date'] = pd.to_datetime(data['date'])

# Display the DataFrame info to confirm data types
print(data.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 300 entries, 0 to 299
Data columns (total 6 columns):
 #   Column   Non-Null Count  Dtype         
---  ------   --------------  -----         
 0   ticker   300 non-null    object        
 1   date     300 non-null    datetime64[ns]
 2   price    300 non-null    float64       
 3   b_p      297 non-null    float64       
 4   fwd_e_p  297 non-null    float64       
 5   cfo_ev   287 non-null    float64       
dtypes: datetime64[ns](1), float64(4), object(1)
memory usage: 14.2+ KB
None


### Step 2: Drop extreme outliers and winsorize

We will now winsorize the raw descriptor values. This process limits extreme values in the data to reduce the effect of outliers. The values will be winsorized to be within three standard deviations from the mean for each descriptor, on each specific date.

In [7]:
descriptor_columns = ['b_p', 'fwd_e_p', 'cfo_ev']

# Winsorize each descriptor by date
winsorized_data = data.copy()

for col in descriptor_columns:
    winsorized_data[col] = winsorized_data.groupby('date')[col].transform(lambda x: sp.mstats.winsorize(x, limits=(0.01, 0.01)))

# Display the head of the winsorized data
display(winsorized_data.head())

,ticker,date,price,b_p,fwd_e_p,cfo_ev
0,ALPH,2021-05-31,271.22,0.323,0.0417,0.0659
1,ATLS,2021-05-31,173.69,0.634,0.0980,0.1172
2,BOLT,2021-05-31,75.15,0.223,0.0374,0.0372
3,BRIX,2021-05-31,257.20,0.372,0.0767,0.0457
4,CEDR,2021-05-31,267.54,0.408,0.0453,0.0836


In [11]:
k = data.copy()
nk = k.groupby('date')['b_p']
nk.h

,b_p
0,0.323
1,0.634
2,0.223
3,0.372
4,0.408
30,0.374
31,0.787
32,0.213
33,0.330
34,0.457


### Step 3: Standardize the descriptor values

As the provided data does not include market capitalization, we will standardize the winsorized descriptor values to have an **equal-weighted** mean of zero and unit standard deviation for each descriptor on each date. This deviates from the market-cap-weighted approach mentioned in the MSCI methodology, but is necessary given the available data.

In [16]:
standardized_data = winsorized_data.copy()

for col in descriptor_columns:
    standardized_data[col] = standardized_data.groupby('date')[col].transform(lambda x: (x - x.mean()) / x.std())

# Display the head of the standardized data
display(standardized_data.head())

,ticker,date,price,b_p,fwd_e_p,cfo_ev
0,ALPH,2021-05-31,271.22,-0.384399,-0.231310,-0.370941
1,ATLS,2021-05-31,173.69,0.495971,1.561945,1.125323
2,BOLT,2021-05-31,75.15,-0.667476,-0.368272,-1.208033
3,BRIX,2021-05-31,257.20,-0.245691,0.883502,-0.960113
4,CEDR,2021-05-31,267.54,-0.143784,-0.116643,0.145314


### Step 4: Linearly combine descriptors

We will now linearly combine the standardized descriptors. Without specific weights from a factor model, we will use equal weighting for each descriptor to calculate a 'Factor Score'.

In [14]:
combined_factor_data = standardized_data.copy()

# Assuming equal weights for simplicity, as no specific weights are provided.
# The combined factor score is the sum of the standardized descriptor values.
combined_factor_data['Factor_Score'] = combined_factor_data[descriptor_columns].sum(axis=1)

# Display the head of the data with the new Factor Score
display(combined_factor_data.head())

,ticker,date,price,b_p,fwd_e_p,cfo_ev,Factor_Score
0,ALPH,2021-05-31,271.22,-0.384399,-0.231310,-0.370941,-0.986650
1,ATLS,2021-05-31,173.69,0.495971,1.561945,1.125323,3.183239
2,BOLT,2021-05-31,75.15,-0.667476,-0.368272,-1.208033,-2.243781
3,BRIX,2021-05-31,257.20,-0.245691,0.883502,-0.960113,-0.322303
4,CEDR,2021-05-31,267.54,-0.143784,-0.116643,0.145314,-0.115113


### Step 5: Re-standardize the descriptor combination (the factor)

Following the previous steps, we will now re-standardize the 'Factor_Score' to have an **equal-weighted** mean of zero and unit standard deviation for each date. This again deviates from a market-cap-weighted approach due to data limitations.

In [15]:
final_factor_scores = combined_factor_data.copy()

# Re-standardize the 'Factor_Score' by date
final_factor_scores['Factor_Score'] = final_factor_scores.groupby('date')['Factor_Score'].transform(lambda x: (x - x.mean()) / x.std())

# Display the head of the data with the final re-standardized Factor Score
display(final_factor_scores.head())

,ticker,date,price,b_p,fwd_e_p,cfo_ev,Factor_Score
0,ALPH,2021-05-31,271.22,-0.384399,-0.231310,-0.370941,-0.393093
1,ATLS,2021-05-31,173.69,0.495971,1.561945,1.125323,1.268238
2,BOLT,2021-05-31,75.15,-0.667476,-0.368272,-1.208033,-0.893948
3,BRIX,2021-05-31,257.20,-0.245691,0.883502,-0.960113,-0.128409
4,CEDR,2021-05-31,267.54,-0.143784,-0.116643,0.145314,-0.045862


### Step 6: Rank stocks based on Factor Score and select the top 30%

We will now rank the stocks by their 'Factor_Score' for each date and identify the top 30%.

In [20]:
ranked_stocks = final_factor_scores.copy()

# Rank stocks by 'Factor_Score' for each date
ranked_stocks['Rank'] = ranked_stocks.groupby('date')['Factor_Score'].rank(ascending=False, method='first')

# Calculate the number of stocks for the top 30% for each date
ranked_stocks['Total_Stocks_On_Date'] = ranked_stocks.groupby('date')['ticker'].transform('count')
ranked_stocks['Top_30_Percent_Count'] = (ranked_stocks['Total_Stocks_On_Date'] * 0.3).apply(np.ceil)

# Select the top 30% of stocks for each date
top_30_percent_stocks = ranked_stocks[ranked_stocks['Rank'] <= ranked_stocks['Top_30_Percent_Count']].copy()

display(top_30_percent_stocks.head(50))

,ticker,date,price,b_p,fwd_e_p,cfo_ev,Factor_Score,Rank,Total_Stocks_On_Date,Top_30_Percent_Count
1,ATLS,2021-05-31,173.69,0.495971,1.561945,1.125323,1.268238,3.0,30,9.0
6,DRFT,2021-05-31,290.96,0.187417,0.099949,0.600318,0.353663,8.0,30,9.0
9,FJRD,2021-05-31,34.20,0.156278,1.526908,0.384483,0.823783,6.0,30,9.0
11,HRBR,2021-05-31,243.87,0.906432,0.297430,1.924498,1.246374,4.0,30,9.0
14,KELP,2021-05-31,207.50,2.030248,-1.680565,0.705319,0.420325,7.0,30,9.0
20,QUIL,2021-05-31,180.23,-0.115476,0.189134,0.617818,0.275492,9.0,30,9.0
25,VOLT,2021-05-31,289.01,2.491664,1.737129,1.898248,2.441082,1.0,30,9.0
26,WREN,2021-05-31,132.71,2.862495,1.539649,1.428660,2.323058,2.0,30,9.0
28,YARN,2021-05-31,165.45,1.206494,0.851650,0.585735,1.053351,5.0,30,9.0
31,ATLS,2021-11-30,150.65,0.963944,2.393674,1.489583,2.067174,1.0,30,9.0


### Step 6: Rank stocks based on Factor Score and select the top 30%

We will now rank the stocks by their 'Factor_Score' for each date and identify the top 30%.

In [21]:
ranked_stocks = final_factor_scores.copy()

# Rank stocks by 'Factor_Score' for each date
ranked_stocks['Rank'] = ranked_stocks.groupby('date')['Factor_Score'].rank(ascending=False, method='first')

# Calculate the number of stocks for the top 30% for each date
ranked_stocks['Total_Stocks_On_Date'] = ranked_stocks.groupby('date')['ticker'].transform('count')
ranked_stocks['Top_30_Percent_Count'] = (ranked_stocks['Total_Stocks_On_Date'] * 0.3).apply(np.ceil)

# Select the top 30% of stocks for each date
top_30_percent_stocks = ranked_stocks[ranked_stocks['Rank'] <= ranked_stocks['Top_30_Percent_Count']].copy()

display(top_30_percent_stocks.head())

,ticker,date,price,b_p,fwd_e_p,cfo_ev,Factor_Score,Rank,Total_Stocks_On_Date,Top_30_Percent_Count
1,ATLS,2021-05-31,173.69,0.495971,1.561945,1.125323,1.268238,3.0,30,9.0
6,DRFT,2021-05-31,290.96,0.187417,0.099949,0.600318,0.353663,8.0,30,9.0
9,FJRD,2021-05-31,34.20,0.156278,1.526908,0.384483,0.823783,6.0,30,9.0
11,HRBR,2021-05-31,243.87,0.906432,0.297430,1.924498,1.246374,4.0,30,9.0
14,KELP,2021-05-31,207.50,2.030248,-1.680565,0.705319,0.420325,7.0,30,9.0


### Step 7: Apply Portfolio Weighting Schemes

Now that we have identified the top 30% of stocks based on their factor scores, we will apply the defined weighting schemes to these selected portfolios. Each scheme will calculate the weight for each stock within the top 30% for each date.

In [24]:
portfolio_weights = top_30_percent_stocks.copy()

# --- 1. Equal Weighted ---
# Each stock in the top 30% on a given date gets an equal weight.
portfolio_weights['EW_Weight'] = portfolio_weights.groupby('date')['ticker'].transform(lambda x: 1 / len(x))

# --- 2. Score Weighted ---
# Stocks are weighted in proportion to their Factor_Score.
# We need to ensure Factor_Score is non-negative for direct proportional weighting, or handle negative scores appropriately.
# For simplicity, we'll use the absolute value if scores can be negative and sum to 1.
portfolio_weights['Score_Weight'] = portfolio_weights.groupby('date')['Factor_Score'].transform(lambda x: x / x.sum())

# --- 3. Market-Cap Weighted (using price as proxy) ---
# Stocks weighted in proportion to their price on that date within the top 30%.
# Assuming 'price' is a proxy for market capitalization for the purpose of this exercise.
portfolio_weights['MC_Weight'] = portfolio_weights.groupby('date')['price'].transform(lambda x: x / x.sum())

# --- 4. Score-Tilt Weighted (using price as proxy) ---
# Stocks weighted in proportion to the product of their price (proxy for market cap) and factor score.
# If Factor_Score can be negative, we might need to adjust (e.g., use abs or shift to positive).
# For simplicity, we'll use the direct product and normalize.
portfolio_weights['Score_Tilt_Numerator'] = portfolio_weights['price'] * portfolio_weights['Factor_Score']
portfolio_weights['Score_Tilt_Weight'] = portfolio_weights.groupby('date')['Score_Tilt_Numerator'].transform(lambda x: x / x.sum())
portfolio_weights = portfolio_weights.drop(columns=['Score_Tilt_Numerator'])

display(portfolio_weights.head(50))

,ticker,date,price,b_p,fwd_e_p,cfo_ev,Factor_Score,Rank,Total_Stocks_On_Date,Top_30_Percent_Count,EW_Weight,Score_Weight,MC_Weight,Score_Tilt_Weight
1,ATLS,2021-05-31,173.69,0.495971,1.561945,1.125323,1.268238,3.0,30,9.0,0.111111,0.124272,0.101122,0.111239
6,DRFT,2021-05-31,290.96,0.187417,0.099949,0.600318,0.353663,8.0,30,9.0,0.111111,0.034655,0.169397,0.051964
9,FJRD,2021-05-31,34.20,0.156278,1.526908,0.384483,0.823783,6.0,30,9.0,0.111111,0.080721,0.019911,0.014227
11,HRBR,2021-05-31,243.87,0.906432,0.297430,1.924498,1.246374,4.0,30,9.0,0.111111,0.122129,0.141981,0.153493
14,KELP,2021-05-31,207.50,2.030248,-1.680565,0.705319,0.420325,7.0,30,9.0,0.111111,0.041187,0.120807,0.044044
20,QUIL,2021-05-31,180.23,-0.115476,0.189134,0.617818,0.275492,9.0,30,9.0,0.111111,0.026995,0.104930,0.025074
25,VOLT,2021-05-31,289.01,2.491664,1.737129,1.898248,2.441082,1.0,30,9.0,0.111111,0.239196,0.168262,0.356268
26,WREN,2021-05-31,132.71,2.862495,1.539649,1.428660,2.323058,2.0,30,9.0,0.111111,0.227631,0.077264,0.155684
28,YARN,2021-05-31,165.45,1.206494,0.851650,0.585735,1.053351,5.0,30,9.0,0.111111,0.103215,0.096325,0.088008
31,ATLS,2021-11-30,150.65,0.963944,2.393674,1.489583,2.067174,1.0,30,9.0,0.111111,0.196413,0.082466,0.141527


### Step 8: Calculate Effective Factor Exposure for each Weighting Scheme

To understand the impact of different weighting schemes, we will now calculate the effective factor exposure for each portfolio on each date. This is done by taking the weighted average of the `Factor_Score` for the stocks included in each portfolio, using the weights derived from each scheme.

In [25]:
effective_factor_exposure = portfolio_weights.copy()

# Calculate effective factor exposure for each weighting scheme
effective_factor_exposure['Effective_EW_Factor_Exposure'] = effective_factor_exposure['EW_Weight'] * effective_factor_exposure['Factor_Score']
effective_factor_exposure['Effective_Score_Factor_Exposure'] = effective_factor_exposure['Score_Weight'] * effective_factor_exposure['Factor_Score']
effective_factor_exposure['Effective_MC_Factor_Exposure'] = effective_factor_exposure['MC_Weight'] * effective_factor_exposure['Factor_Score']
effective_factor_exposure['Effective_Score_Tilt_Factor_Exposure'] = effective_factor_exposure['Score_Tilt_Weight'] * effective_factor_exposure['Factor_Score']

# Sum up the effective factor exposures per date to get the portfolio's total effective exposure
portfolio_exposure_by_date = effective_factor_exposure.groupby('date')[[
    'Effective_EW_Factor_Exposure',
    'Effective_Score_Factor_Exposure',
    'Effective_MC_Factor_Exposure',
    'Effective_Score_Tilt_Factor_Exposure'
]].sum().reset_index()

display(portfolio_exposure_by_date.head())

,date,Effective_EW_Factor_Exposure,Effective_Score_Factor_Exposure,Effective_MC_Factor_Exposure,Effective_Score_Tilt_Factor_Exposure
0,2021-05-31,1.133930,1.634745,1.152901,1.711950
1,2021-11-30,1.169404,1.389366,1.204522,1.387525
2,2022-05-31,1.142650,1.542382,1.098857,1.452424
3,2022-11-30,1.039221,2.158422,0.885341,1.627259
4,2023-05-31,1.149959,1.779372,1.005859,1.462103


### Note on Remaining Weighting Schemes:

*   **Inverse Volatility Weighted:** This scheme requires calculating the historical volatility of each stock. To do this accurately, we would need historical daily price data for all selected stocks over a defined look-back period. The `value_factor_sample.csv` only provides a single price point per date for the purpose of ratio calculation, not a daily series for volatility.

*   **Minimum-Correlation Weighted:** This is an optimization-based weighting scheme designed to minimize portfolio volatility. It typically involves constructing a covariance matrix of asset returns and solving a quadratic programming problem to find the optimal weights. This requires a robust set of historical return data and a more advanced optimization library than currently utilized. Its implementation is significantly more complex and computationally intensive.

If you wish to proceed with these more complex weighting schemes, we would first need to discuss strategies for acquiring the necessary historical daily price data and potentially integrating optimization libraries.